## Streamlined methods to copy the SDE sample design tracking data and prepare for the HAF proportional area analysis

Author: Kaitlin Lubetkin, Alex Traynor
Created: 12/26/23  
Last Edited: 09/06/24  

## 1) Initiate variables 
<code style="background:yellow;color:black">***(edit for specific analysis)***</code>

In [11]:
import os

# Parent/root folder for analysis
parentFolder = (r'C:\Users\alaurencetraynor\Documents\Tools')
# Analysis name
analysis_name = "example"
aoi = "Benchmark Groups"
# SDD geodatabase - this should be a copy you'll use for your analysis
sddsde = os.path.join(parentFolder, ("SDDSDE_"+ analysis_name +".gdb"))

## 2) Populate SDD geodatabase
<code style="background:yellow;color:black">***(run straight through, no edits needed)***</code>

In [12]:
if not arcpy.Exists(sddsde):
    arcpy.management.CreateFileGDB(out_folder_path=os.path.dirname(sddsde), 
                                   out_name=os.path.basename(sddsde))
arcpy.env.workspace = sddsde
    
# Copy TerrADat and LMF design data from the SDE to the analysis SDD geodatabase, 
# clipping to the fine-scale
sde = (r'\\blm\dfs\loc\EGIS\ProjectsNational\AIM\AIMDataTools\SDE\AIMDev.sde' + 
       r'\ilmocAIMdev.ILMNATAIMDEV.')
to_copy = ["LMF", "LMFNotSampled", "LMFDesignPoints", "LMFThiessenPolygons", "LMFSegmentPolygons", 
           "TerraDat", "NotSampled", "DesignPoints", "DesignPolygons"]
for fc in to_copy:
    arcpy.analysis.PairwiseClip(
        in_features=sde + fc,
        clip_features=aoi,
        out_feature_class=sddsde + "\\" + fc)

# Add the TerrADat relationship classes
for tbl in ["DesignPoints", "TerraDat", "NotSampled"]:
    arcpy.management.CreateRelationshipClass(origin_table="DesignPolygons", 
                                             destination_table=tbl, 
                                             out_relationship_class="DesignPolygonsTo" + tbl, 
                                             relationship_type="SIMPLE", 
                                             forward_label=tbl, 
                                             backward_label="DesignPolygons", 
                                             cardinality="ONE_TO_MANY", 
                                             origin_primary_key="DesignPolygonID", 
                                             origin_foreign_key="DesignPolygonID")
for tbl in ["TerraDat", "NotSampled"]:
    arcpy.management.CreateRelationshipClass(origin_table="DesignPoints", 
                                             destination_table=tbl, 
                                             out_relationship_class="DesignPointsTo" + tbl, 
                                             relationship_type="SIMPLE", 
                                             forward_label=tbl, 
                                             backward_label="DesignPoints", 
                                             cardinality="ONE_TO_MANY", 
                                             origin_primary_key="DesignPointKey", 
                                             origin_foreign_key="DesignPointKey")
print("Copied")

ExecuteError: Failed to execute. Parameters are not valid.
ERROR 000732: Input Features: Dataset \\blm\dfs\loc\EGIS\ProjectsNational\AIM\AIMDataTools\SDE\AIMDev.sde\ilmocAIMdev.ILMNATAIMDEV.LMF does not exist or is not supported
Failed to execute (PairwiseClip).


In [ ]:
# Clean Arc Pro workspace
map = arcpy.mp.ArcGISProject("CURRENT").listMaps("Map")[0]
# Remove the newly created LMF strata and segment layers from the map
for lyr in map.listLayers():
    #print(lyr.name)
    if lyr.name in to_copy:
        map.removeLayer(lyr)

print("Done!")